# Président — baseline CamemBERT **sans chunks**

Ce notebook entraîne un classifieur **phrase par phrase** (sans chunking), avec :

- split **par document** en `train / val / test_local`
- CamemBERT pour classification binaire
- évaluation sur :
  - validation phrases
  - test local phrases
- sauvegarde du meilleur modèle
- génération optionnelle d'une soumission finale

Labels :
- `C = 0` (Chirac)
- `M = 1` (Mitterrand)

La probabilité de sortie est donc **P(Mitterrand)**.


In [2]:

import os

# =========================
# 0) Configuration
# =========================

# Remplace ces chemins si besoin (Colab / Drive)
TRAIN_FILE = "/content/drive/MyDrive/projet tal/corpus.tache1.learn.utf8"
#TEST_FILE  = "corpus.tache1.test.utf8"   # optionnel pour la soumission finale

MODEL_NAME = "camembert/camembert-large"

MODEL_SAVE_DIR = "/content/drive/MyDrive/projet tal/rital_tache-doc/models/v14_corrige_best"
OUTPUT_DIR     = "/content/drive/MyDrive/projet tal/rital_tache-doc/runs/v14_corrige"
SUBMISSION_DIR = "/content/drive/MyDrive/projet tal/rital_tache-doc/submissions"

# Split local
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10

# Entraînement
MAX_LENGTH = 256
BATCH_SIZE = 8
GRAD_ACCUM = 4
LR = 1e-5
NUM_EPOCHS = 6
WEIGHT_DECAY = 0.01
SEED = 42

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

print("MODEL_SAVE_DIR =", MODEL_SAVE_DIR)
print("OUTPUT_DIR     =", OUTPUT_DIR)
print("SUBMISSION_DIR =", SUBMISSION_DIR)


MODEL_SAVE_DIR = /content/drive/MyDrive/projet tal/rital_tache-doc/models/v14_corrige_best
OUTPUT_DIR     = /content/drive/MyDrive/projet tal/rital_tache-doc/runs/v14_corrige
SUBMISSION_DIR = /content/drive/MyDrive/projet tal/rital_tache-doc/submissions


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

# =========================
# 1) Imports
# =========================
import re
import json
import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

import torch
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)


In [4]:

# =========================
# 2) Seed
# =========================
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)


device = cuda


In [5]:

# =========================
# 3) Lecture du corpus
# =========================

LINE_RE = re.compile(r"^<(\d+):(\d+):([CM])>\s*(.*)$")

def read_labeled_corpus(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.rstrip("\n")
            if not line.strip():
                continue
            m = LINE_RE.match(line)
            if not m:
                raise ValueError(f"Ligne mal formée à la ligne {line_num}: {line[:200]}")
            doc_id = int(m.group(1))
            sent_id = int(m.group(2))
            lab = m.group(3)
            text = m.group(4).strip()
            rows.append({
                "doc_id": doc_id,
                "sent_id": sent_id,
                "label_str": lab,
                "label": 1 if lab == "M" else 0,
                "text": text
            })
    df = pd.DataFrame(rows).sort_values(["doc_id", "sent_id"]).reset_index(drop=True)
    return df

df = read_labeled_corpus(TRAIN_FILE)
print(df.head())
print()
print("Nb phrases:", len(df))
print("Nb documents:", df["doc_id"].nunique())
print("Répartition labels:", df["label_str"].value_counts().to_dict())


   doc_id  sent_id label_str  label  \
0       3        1         C      0   
1       3        2         C      0   
2       3        3         C      0   
3       3        4         C      0   
4       3        5         C      0   

                                                text  
0  Je voudrais d'abord vous remercier, Madame la ...  
1  Et croyez que c'est une amitié, dans mon coeur...  
2  Je ressens avec une émotion particulière l'hon...  
3  Je veux y voir le signe de l'amitié entre deux...  
4  Cette Europe fut longtemps le champ de bataill...  

Nb phrases: 57413
Nb documents: 587
Répartition labels: {'C': 49890, 'M': 7523}


In [6]:

# =========================
# 4) Diagnostic corpus
# =========================
doc_label_sets = df.groupby("doc_id")["label_str"].apply(lambda s: set(s.tolist()))
n_mixed = int((doc_label_sets.apply(len) > 1).sum())

print("Documents mixtes:", n_mixed, "/", df["doc_id"].nunique())

doc_stats = df.groupby("doc_id").agg(
    n_phrases=("text", "size"),
    n_M=("label", "sum"),
)
doc_stats["n_C"] = doc_stats["n_phrases"] - doc_stats["n_M"]
doc_stats.describe()


Documents mixtes: 400 / 587


,n_phrases,n_M,n_C
count,587.000000,587.000000,587.000000
mean,97.807496,12.816014,84.991482
std,59.041740,9.848869,61.147655
min,8.000000,0.000000,4.000000
25%,54.000000,0.000000,39.000000
50%,81.000000,15.000000,66.000000
75%,132.000000,20.500000,119.000000
max,393.000000,39.000000,376.000000


In [7]:

# =========================
# 5) Split par document
# =========================

doc_df = df.groupby("doc_id").agg(
    n_phrases=("text", "size"),
    n_M=("label", "sum"),
).reset_index()
doc_df["has_M"] = (doc_df["n_M"] > 0).astype(int)

train_docs, temp_docs = train_test_split(
    doc_df["doc_id"],
    test_size=(1.0 - TRAIN_RATIO),
    random_state=SEED,
    stratify=doc_df["has_M"],
)

temp_df = doc_df[doc_df["doc_id"].isin(temp_docs)].copy()
val_relative = VAL_RATIO / (VAL_RATIO + TEST_RATIO)

val_docs, test_docs = train_test_split(
    temp_df["doc_id"],
    test_size=(1.0 - val_relative),
    random_state=SEED,
    stratify=temp_df["has_M"],
)

train_docs = set(train_docs.tolist())
val_docs   = set(val_docs.tolist())
test_docs  = set(test_docs.tolist())

train_df = df[df["doc_id"].isin(train_docs)].copy().reset_index(drop=True)
val_df   = df[df["doc_id"].isin(val_docs)].copy().reset_index(drop=True)
test_local_df = df[df["doc_id"].isin(test_docs)].copy().reset_index(drop=True)

def summarize_split(name, xdf):
    n_docs = xdf["doc_id"].nunique()
    n_rows = len(xdf)
    counts = xdf["label_str"].value_counts().to_dict()
    print(f"{name}: docs={n_docs}, phrases={n_rows}, labels={counts}")

summarize_split("train", train_df)
summarize_split("val", val_df)
summarize_split("test_local", test_local_df)


train: docs=469, phrases=46079, labels={'C': 40077, 'M': 6002}
val: docs=59, phrases=5988, labels={'C': 5263, 'M': 725}
test_local: docs=59, phrases=5346, labels={'C': 4550, 'M': 796}


In [8]:

# =========================
# 6) Dataset phrase par phrase
# =========================

def make_sentence_entries(xdf):
    return xdf[["doc_id", "sent_id", "text", "label"]].to_dict(orient="records")

train_entries = make_sentence_entries(train_df)
val_entries = make_sentence_entries(val_df)
test_local_entries = make_sentence_entries(test_local_df)

print("train_entries:", len(train_entries))
print("val_entries:", len(val_entries))
print("test_local_entries:", len(test_local_entries))


train_entries: 46079
val_entries: 5988
test_local_entries: 5346


In [9]:

# =========================
# 7) Tokenizer
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/809k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/374 [00:00<?, ?B/s]

In [10]:

# =========================
# 8) Hugging Face datasets
# =========================

train_ds = Dataset.from_pandas(pd.DataFrame(train_entries))
val_ds = Dataset.from_pandas(pd.DataFrame(val_entries))
test_local_ds = Dataset.from_pandas(pd.DataFrame(test_local_entries))

remove_cols = [c for c in train_ds.column_names if c not in ["text", "label"]]

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=remove_cols + ["text"])
val_tok = val_ds.map(tokenize_batch, batched=True, remove_columns=remove_cols + ["text"])
test_local_tok = test_local_ds.map(tokenize_batch, batched=True, remove_columns=remove_cols + ["text"])

print(train_tok)
print(val_tok)
print(test_local_tok)


Map:   0%|          | 0/46079 [00:00<?, ? examples/s]

Map:   0%|          | 0/5988 [00:00<?, ? examples/s]

Map:   0%|          | 0/5346 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 46079
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 5988
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 5346
})


In [11]:

# =========================
# 9) Métriques
# =========================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)

    out = {
        "f1_macro": f1_score(labels, preds, average="macro") * 100,
        "auc": roc_auc_score(labels, probs) * 100,
        "ap": average_precision_score(labels, probs) * 100,
    }
    return out


In [12]:

# =========================
# 10) Modèle
# =========================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)


model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: camembert/camembert-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:

# =========================
# 11) TrainingArguments
# =========================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="auc",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)


In [14]:

# =========================
# 12) Trainer
# =========================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


In [15]:

# =========================
# 13) Entraînement
# =========================

train_result = trainer.train()
metrics = trainer.evaluate()

print("BEST CHECKPOINT (validation phrases):")
for k, v in metrics.items():
    if k.startswith("eval_"):
        name = k.replace("eval_", "")
        print(f"{name:>14} : {v}")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 6, 'bos_token_id': 5}.


Epoch,Training Loss,Validation Loss,F1 Macro,Auc,Ap
1,0.868462,0.243154,76.664277,92.871536,75.249236
2,0.629171,0.198331,82.505396,94.107622,78.031977
3,0.419392,0.319139,82.565882,93.239545,77.318677
4,0.290785,0.407213,80.898513,91.646694,75.289395


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

BEST CHECKPOINT (validation phrases):
          loss : 0.19925232231616974
      f1_macro : 82.5129259667806
           auc : 94.09510768081661
            ap : 78.01905820701192
       runtime : 26.8592
samples_per_second : 222.94
steps_per_second : 27.886


In [16]:

# =========================
# 14) Sauvegarde du meilleur modèle
# =========================

trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)

with open(os.path.join(MODEL_SAVE_DIR, "config_run.json"), "w", encoding="utf-8") as f:
    json.dump({
        "train_file": TRAIN_FILE,
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "lr": LR,
        "num_epochs_max": NUM_EPOCHS,
        "seed": SEED,
        "mode": "phrase_par_phrase",
    }, f, ensure_ascii=False, indent=2)

print("Modèle sauvegardé dans:", MODEL_SAVE_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle sauvegardé dans: /content/drive/MyDrive/projet tal/rital_tache-doc/models/v14_corrige_best


In [17]:

# =========================
# 15) Évaluation sur test local
# =========================

pred_output = trainer.predict(test_local_tok)
logits = pred_output.predictions
y_true = np.array(test_local_df["label"].tolist())

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
y_pred = (probs >= 0.5).astype(int)

f1m = f1_score(y_true, y_pred, average="macro") * 100
auc = roc_auc_score(y_true, probs) * 100
ap = average_precision_score(y_true, probs) * 100

print(f"[test_local_phrases] F1_macro={f1m:.2f} | AUC={auc:.2f} | AP={ap:.2f}")

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix [[TN, FP],[FN, TP]]:")
print(cm)

print(classification_report(
    y_true,
    y_pred,
    target_names=["Chirac", "Mitterrand"],
    digits=4
))


[test_local_phrases] F1_macro=81.53 | AUC=94.25 | AP=81.01
Confusion matrix [[TN, FP],[FN, TP]]:
[[4480   70]
 [ 354  442]]
              precision    recall  f1-score   support

      Chirac     0.9268    0.9846    0.9548      4550
  Mitterrand     0.8633    0.5553    0.6758       796

    accuracy                         0.9207      5346
   macro avg     0.8950    0.7699    0.8153      5346
weighted avg     0.9173    0.9207    0.9133      5346



In [18]:

# =========================
# 16) Distribution des probabilités
# =========================

print("Distribution des probabilités:")
print(f"  < 0.1:     {(probs < 0.1).sum()} ({(probs < 0.1).mean()*100:.1f}%)")
print(f"  > 0.9:     {(probs > 0.9).sum()} ({(probs > 0.9).mean()*100:.1f}%)")
mid = ((probs >= 0.1) & (probs <= 0.9))
print(f"  0.1-0.9:   {mid.sum()} ({mid.mean()*100:.1f}%)")


Distribution des probabilités:
  < 0.1:     4504 (84.2%)
  > 0.9:     334 (6.2%)
  0.1-0.9:   508 (9.5%)


In [ ]:

# =========================
# 17) Lecture du vrai fichier test (optionnel)
# =========================

TEST_RE = re.compile(r"^<(\d+):(\d+)>\s*(.*)$")

def read_unlabeled_test(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.rstrip("\n")
            if not line.strip():
                continue
            m = TEST_RE.match(line)
            if not m:
                raise ValueError(f"Ligne test mal formée à la ligne {line_num}: {line[:200]}")
            rows.append({
                "doc_id": int(m.group(1)),
                "sent_id": int(m.group(2)),
                "text": m.group(3).strip(),
            })
    return pd.DataFrame(rows).sort_values(["doc_id", "sent_id"]).reset_index(drop=True)

if os.path.exists(TEST_FILE):
    test_df = read_unlabeled_test(TEST_FILE)
    print(test_df.head())
    print("Nb lignes test:", len(test_df))
else:
    print("TEST_FILE introuvable pour l'instant :", TEST_FILE)


In [ ]:

# =========================
# 18) Prédiction finale sur le vrai test (optionnel)
# =========================

if 'test_df' in globals():
    final_test_ds = Dataset.from_pandas(test_df.copy())
    remove_cols = [c for c in final_test_ds.column_names if c != "text"]
    final_test_tok = final_test_ds.map(tokenize_batch, batched=True, remove_columns=remove_cols + ["text"])

    final_pred = trainer.predict(final_test_tok)
    final_probs = torch.softmax(torch.tensor(final_pred.predictions), dim=1).numpy()[:, 1]

    submission_path = os.path.join(SUBMISSION_DIR, "submission-pres-phrase-baseline.csv")
    pd.DataFrame(final_probs).to_csv(submission_path, index=False, header=False)

    print("Soumission sauvegardée dans :", submission_path)
    print("Aperçu :")
    print(pd.DataFrame(final_probs).head())
else:
    print("Pas de test chargé.")
